# Génération de population — pipeline multi-taille avec checkpoints

Ce notebook orchestre la création de populations synthétiques pour plusieurs tailles cibles,
avec checkpoints intermédiaires dans un dossier `Temp/` situé à côté du notebook.

## Paramètres

- `POPULATION_SIZES` : liste des tailles à générer (multiples de 100)
- `FORCE_REGENERATE` : force la régénération depuis eqasim (ignore le cache de l'API)
- `FORCE_STEP` : force la reprise à partir d'une étape — `raw`, `fixed`, `pt_enriched`, `zone_enriched` ou `scheduled`

## Pipeline avec checkpoints

| Étape | Entrée | Sortie |
|---|---|---|
| 1 – Génération eqasim | API eqasim | `Temp/1_raw/` |
| 2 – Validation activités | `Temp/1_raw/` | `Temp/2_fixed/` |
| 3 – Enrichissement PT | `Temp/2_fixed/` | `Temp/3_pt_enriched/` |
| 3bis – Enrichissement zone (AAV2020 + densité) | `Temp/3_pt_enriched/` | `Temp/4_zone_enriched/` |
| 4 – Calcul itinéraires OSMnx | `Temp/4_zone_enriched/` | `Temp/5_scheduled/` |
| Export final | `Temp/5_scheduled/` | `data/eqasim_output/` |

À chaque étape, si le fichier de sortie existe déjà dans `Temp/`, l'étape est ignorée.
Pour forcer la reprise à une étape donnée (et toutes les suivantes), définir `FORCE_STEP`.

## Prérequis

| Service | Commande | Port |
|---|---|---|
| eqasim | `docker compose up eqasim` | 8003 |

Dépendances Python : `numpy`, `pandas`, `tqdm`, `osmnx`, `python-calamine`

### Données INSEE requises (étape 3bis)

Placer dans `data/insee/` :

| Fichier | Source | Rôle |
|---|---|---|
| `fichier_diffusion_2026.xlsx` | [insee.fr/fr/statistiques/5040028](https://www.insee.fr/fr/statistiques/5040028) | Grille densité 2025 — colonne `DENS_AAV` (requis) |
| `AAV2020_au_01-01-2026.xlsx` | [insee.fr/fr/statistiques/5040879](https://www.insee.fr/fr/statistiques/5040879) | Catégorie pôle/couronne par commune (optionnel) |

La colonne `DENS_AAV` du fichier diffusion contient déjà le croisement densité × AAV :
`1=Urbain dense` · `2=Urbain intermédiaire` · `3=Rural périurbain` · `4=Rural non périurbain`

In [ ]:
import os, certifi
os.environ['SSL_CERT_FILE'] = certifi.where()
os.environ['REQUESTS_CA_BUNDLE'] = certifi.where()

In [ ]:
# ── Paramètres ────────────────────────────────────────────────────────────────
POPULATION_SIZES      = [10]      # multiples de 100 ; peut atteindre des centaines de milliers
GENERATE_PERSONALITY  = False           # True → génère les Big Five (lent)
FORCE_REGENERATE      = False           # True → appelle eqasim avec force=True + reécrit Temp/raw/
FORCE_STEP            = None            # None | 'raw' | 'fixed' | 'pt_enriched' | 'zone_enriched' | 'scheduled'
                                        # Force la reprise à partir de cette étape (et toutes les suivantes)

BBOX = None                               # [min_lon, min_lat, max_lon, max_lat] WGS84 — None = dept 31
# BBOX = [1.35, 43.55, 1.50, 43.65]      # exemple : centre de Toulouse

CLEAR_DOWNSTREAM_ON_REGENERATE = True    # True → supprime les checkpoints downstream quand l'étape 1 régénère un raw

EQASIM_URL = 'http://localhost:8003'

## Initialisation de l'environnement

Création de l'arborescence de dossiers temporaires (`Temp/`) et définition des chemins vers les données sources et de sortie. Chaque étape écrit dans un sous-dossier numéroté (`1_raw/`, `2_fixed/`, …) pour permettre la reprise depuis n'importe quel point sans tout recalculer.

La fonction `should_force(step)` détermine si une étape doit être rejouée en tenant compte de `FORCE_REGENERATE` et `FORCE_STEP`.

In [ ]:
# ── Chemins & dossiers Temp ───────────────────────────────────────────────────
import json
import os
import sys
import time
import urllib.request
import urllib.error
from pathlib import Path

REPO_ROOT    = Path('../../../').resolve()
POP_DIR      = REPO_ROOT / 'data' / 'population'
NOTEBOOK_DIR = Path('.').resolve()
TEMP_DIR     = NOTEBOOK_DIR / 'Temp'

TEMP_RAW       = TEMP_DIR / '1_raw'
TEMP_FIXED     = TEMP_DIR / '2_fixed'
TEMP_PT        = TEMP_DIR / '3_pt_enriched'
TEMP_ZONE      = TEMP_DIR / '4_zone_enriched'
TEMP_SCHEDULED = TEMP_DIR / '5_scheduled'

for d in [POP_DIR, TEMP_RAW, TEMP_FIXED, TEMP_PT, TEMP_ZONE, TEMP_SCHEDULED]:
    d.mkdir(parents=True, exist_ok=True)

# ── Utilitaires ───────────────────────────────────────────────────────────────
STEPS = ['raw', 'fixed', 'pt_enriched', 'zone_enriched', 'scheduled']

def should_force(step_name: str) -> bool:
    if FORCE_REGENERATE and step_name == 'raw':
        return True
    if FORCE_STEP is None or FORCE_STEP not in STEPS:
        return False
    return STEPS.index(step_name) >= STEPS.index(FORCE_STEP)

def pop_filename(n: int) -> str:
    return f'toulouse_population_{n}.json'

def save_json(data, path: Path) -> None:
    tmp = path.with_suffix('.json.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    os.rename(tmp, path)

def load_json(path: Path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

def check_temporal_order(data: list) -> tuple[int, int]:
    """Check start_time ordering (at most 1 decreasing gap allowed for midnight wrap).
    Returns (n_persons_with_error, n_total_errors).
    """
    n_persons, n_total = 0, 0
    for person in data:
        acts = person.get('identity', {}).get('activities', [])
        n = len(acts)
        if n == 0:
            continue
        errors = []
        for i, act in enumerate(acts):
            s, e = act.get('start_time'), act.get('end_time')
            if s is not None and e is not None and (e - s) % 86400 == 0:
                errors.append(f"act[{i}] durée nulle")
        nb_ecarts = sum(
            1 for i in range(n)
            if acts[(i - 1) % n].get('start_time', 0) > acts[i].get('start_time', 0)
        )
        if nb_ecarts > 1:
            errors.append(f"{nb_ecarts} écarts décroissants sur start_time")
        if errors:
            n_persons += 1
            n_total += len(errors)
    return n_persons, n_total

print(f'REPO_ROOT : {REPO_ROOT}')
print(f'POP_DIR   : {POP_DIR}')
print(f'TEMP_DIR  : {TEMP_DIR}')
print(f'Tailles   : {POPULATION_SIZES}')

## Chargement des dépendances scientifiques

Import des bibliothèques de calcul numérique (`numpy`, `pandas`) et configuration des constantes partagées entre les étapes :

| Constante | Valeur | Rôle |
|---|---|---|
| `GTFS_STOPS` | `data/gtfs/tisseo_gtfs/stops.txt` | Arrêts Tisséo pour l'enrichissement TC |
| `OSMNX_CACHE` | `data/cache/osmnx/` | Cache des graphes routiers (évite le re-téléchargement) |
| `MAX_WORKERS` | 12 | Parallélisme du calcul de routes |
| `MAX_PT_DIST_M` | 1 500 m | Rayon de rattachement à un arrêt TC |

In [ ]:
# ── Imports scientifiques & constantes ───────────────────────────────────────
import hashlib
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

GTFS_STOPS     = REPO_ROOT / 'data' / 'gtfs' / 'tisseo_gtfs' / 'stops.txt'
OSMNX_CACHE       = REPO_ROOT / 'data' / 'cache' / 'osmnx'      # graphs (pickle) — local notebook
OSMNX_ROUTE_CACHE = REPO_ROOT / 'data' / 'cache' / 'osmnx'             # SQLite routes — chemin hôte
SCRIPTS_POP    = REPO_ROOT / 'scripts' / 'data' / 'population'
LLMAGENTS_PATH = str(REPO_ROOT / 'llm-agents')

CACHE_KEY     = hashlib.md5(b'Toulouse, France_30000').hexdigest()[:12]
MAX_WORKERS   = 12
MAX_PT_DIST_M = 1500.0

if LLMAGENTS_PATH not in sys.path:
    sys.path.insert(0, LLMAGENTS_PATH)
if str(SCRIPTS_POP) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_POP))

from population_utils import TRIP_MODES

print(f'GTFS_STOPS:  {GTFS_STOPS}')
print(f'OSMNX_CACHE: {OSMNX_CACHE}')
print(f'OSMnx cache key: {CACHE_KEY}')
print(f'OSMnx route SQLite: {OSMNX_ROUTE_CACHE}')

## Vérification du service eqasim

Avant tout traitement, on s'assure que le service de génération de population eqasim répond sur `EQASIM_URL` (port 8003). En cas d'échec après 3 tentatives, le pipeline s'arrête immédiatement avec un message d'erreur explicite.

> **Prérequis** : `docker compose up eqasim` doit être lancé avant d'exécuter cette cellule.

In [ ]:
# ── Vérification santé du service eqasim ─────────────────────────────────────
def check_health(url: str, retries: int = 3, delay: float = 2.0) -> bool:
    for i in range(retries):
        try:
            with urllib.request.urlopen(f'{url}/health', timeout=5) as r:
                return r.status == 200
        except Exception as e:
            print(f'  tentative {i+1}/{retries} : {e}')
            if i < retries - 1:
                time.sleep(delay)
    return False

if check_health(EQASIM_URL):
    print(f'Service eqasim OK → {EQASIM_URL}')
else:
    raise RuntimeError(
        f'Service eqasim inaccessible sur {EQASIM_URL}. '
        'Démarrez le service avec : docker compose up eqasim'
    )

---
## Étape 1 — Génération eqasim → `Temp/1_raw/`

Appel à l'API eqasim pour chaque taille de population définie dans `POPULATION_SIZES`. eqasim synthétise des agents toulousains avec leurs activités quotidiennes (domicile, travail, loisirs…) à partir de données INSEE et d'enquêtes de déplacements (EMC²).

- **Cache** : si le fichier existe déjà dans `Temp/1_raw/` et que `FORCE_REGENERATE=False`, l'appel API est ignoré.
- **Sortie** : un fichier JSON par taille contenant la liste des personnes avec leurs activités brutes et leurs attributs socio-démographiques.
- **Validation** : après génération, l'ordre temporel des activités est vérifié — au plus un écart décroissant est toléré (passage minuit).

In [ ]:
# ── Étape 1 — Génération eqasim → Temp/1_raw/ ────────────────────────────────
print('=' * 60)
print('ÉTAPE 1 — Génération eqasim → Temp/1_raw/')
print('=' * 60)

for pop_size in POPULATION_SIZES:
    fname    = pop_filename(pop_size)
    raw_path = TEMP_RAW / fname

    if not should_force('raw') and raw_path.exists():
        size_mb = raw_path.stat().st_size / 1_048_576
        print(f'[SKIP] {fname}  ({size_mb:.1f} Mo) — déjà dans Temp/1_raw/')
        continue

    payload = {
        'population_size':      pop_size,
        'generate_personality': GENERATE_PERSONALITY,
        'force':                FORCE_REGENERATE,
    }
    if BBOX is not None:
        payload['bbox'] = BBOX

    body = json.dumps(payload).encode()
    req  = urllib.request.Request(
        f'{EQASIM_URL}/generate',
        data=body,
        headers={'Content-Type': 'application/json'},
        method='POST',
    )

    print(f'[GEN]  {fname}  (population_size={pop_size})…')
    t0 = time.monotonic()

    try:
        with urllib.request.urlopen(req, timeout=7200) as resp:
            result = json.loads(resp.read())
    except urllib.error.HTTPError as e:
        result = json.loads(e.read())
        raise RuntimeError(f'eqasim HTTP {e.code} : {result}')

    elapsed = time.monotonic() - t0
    if result.get('status') != 'ok':
        raise RuntimeError(f'Échec eqasim pour {pop_size} agents : {result}')

    eqasim_out = POP_DIR / fname
    if not eqasim_out.exists():
        raise FileNotFoundError(f'Fichier eqasim introuvable : {eqasim_out}')

    data = load_json(eqasim_out)
    save_json(data, raw_path)
    size_mb = raw_path.stat().st_size / 1_048_576
    print(f'       {len(data)} personnes en {elapsed:.1f}s  ({size_mb:.1f} Mo) → Temp/1_raw/{fname}')
    if CLEAR_DOWNSTREAM_ON_REGENERATE:
        for _dl_dir in [TEMP_FIXED, TEMP_PT, TEMP_SCHEDULED]:
            _p = _dl_dir / fname
            if _p.exists():
                _p.unlink()
                print(f'       [CASCADE] Supprimé {_dl_dir.name}/{fname}')
        _sqlite = OSMNX_CACHE / f'toulouse_population_{pop_size}' / 'osmnx_cache.db'
        if _sqlite.exists():
            _sqlite.unlink()
            print(f'       [CASCADE] Supprimé cache SQLite OSMnx : {_sqlite.relative_to(REPO_ROOT)}')
    n_p, n_e = check_temporal_order(data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(data) - n_p}/{len(data)} personnes valides')

print()
print('Étape 1 terminée.')

---
## Étape 2 — Validation & correction des activités → `Temp/2_fixed/`

Les données brutes eqasim peuvent contenir des séquences d'activités invalides (chevauchements, durées nulles, enchaînements incohérents). Cette étape applique deux passes de nettoyage :

1. **Correction des violations de séquence** via `fix_activities` : réajustement des bornes temporelles pour éliminer chevauchements et durées négatives.
2. **Fusion des activités redondantes** : deux activités consécutives (ou circulaires) de même `purpose` et même localisation sont fusionnées en une seule, pour éviter des micro-trajets de durée nulle.

Chaque agent est re-validé après correction ; les cas encore invalides sont signalés mais n'interrompent pas le pipeline.

In [ ]:
# ── Étape 2 — Validation & correction des activités → Temp/2_fixed/ ──────────
from population_utils import check_activities, fix_activities

print('=' * 60)
print('ÉTAPE 2 — Validation & correction des activités → Temp/2_fixed/')
print('=' * 60)

def _same_loc(a, b):
    la, lb = a.get('location'), b.get('location')
    if la is None or lb is None:
        return la is lb
    return la.get('lon') == lb.get('lon') and la.get('lat') == lb.get('lat')

for pop_size in POPULATION_SIZES:
    fname      = pop_filename(pop_size)
    raw_path   = TEMP_RAW   / fname
    fixed_path = TEMP_FIXED / fname

    if not should_force('fixed') and fixed_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/2_fixed/')
        continue

    if not raw_path.exists():
        raise FileNotFoundError(f'Fichier raw manquant — relancer étape 1 : {raw_path}')

    data = load_json(raw_path)

    # Correction des violations de séquence
    persons_fixed = 0
    fixed_data = []
    for person in data:
        fixed_person, _ = fix_activities(person)
        remaining = check_activities(fixed_person)
        if remaining:
            print(f'  !! person {person.get("person_id", "?")} : encore invalide après correction : {remaining}')
        fixed_data.append(fixed_person)
        persons_fixed += 1

    # Fusion : activités consécutives ET circulaires partageant même purpose ET même location
    n_merged = 0
    for person in fixed_data:
        acts = person.get('identity', {}).get('activities', [])

        # Fusion consécutive
        i = 0
        while i < len(acts) - 1:
            if acts[i].get('purpose') == acts[i + 1].get('purpose') and _same_loc(acts[i], acts[i + 1]):
                acts[i]['end_time'] = acts[i + 1]['end_time']
                acts[i]['scheduled_start_time'] = None
                acts.pop(i + 1)
                n_merged += 1
            else:
                i += 1

        # Fusion circulaire : première et dernière
        if len(acts) >= 2 and acts[0].get('purpose') == acts[-1].get('purpose') and _same_loc(acts[0], acts[-1]):
            last = acts.pop()
            acts[0]['start_time'] = last['start_time']
            acts[0]['scheduled_start_time'] = None
            n_merged += 1

    save_json(fixed_data, fixed_path)
    print(f'[OK]   {fname} — {persons_fixed} corrigé(s), {n_merged} fusion(s) → Temp/2_fixed/')
    n_p, n_e = check_temporal_order(fixed_data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(fixed_data) - n_p}/{len(fixed_data)} personnes valides')

print()
print('Étape 2 terminée.')

---
## Étape 3 — Enrichissement transports en commun → `Temp/3_pt_enriched/`

Pour chaque localisation d'activité, on calcule si elle se trouve à moins de `MAX_PT_DIST_M` (1 500 m) d'un arrêt Tisséo. Ce flag `public_transport` est ensuite utilisé par l'agent LLM pour décider du mode de déplacement pertinent (marche, voiture, TC…).

- **Source** : fichier `stops.txt` du GTFS Tisséo — 5 661 arrêts chargés en mémoire.
- **Méthode** : recherche du plus proche arrêt par distance euclidienne approximative sur les coordonnées WGS84.
- Le nombre de localisations enrichies est affiché par fichier.

In [ ]:
# ── Étape 3 — Enrichissement des flags public_transport → Temp/3_pt_enriched/ ─
from population_utils import enrich_public_transport

print('=' * 60)
print('ÉTAPE 3 — Enrichissement public_transport → Temp/3_pt_enriched/')
print('=' * 60)

stops_df  = pd.read_csv(GTFS_STOPS, usecols=['stop_lat', 'stop_lon'])
stop_lats = stops_df['stop_lat'].values
stop_lons = stops_df['stop_lon'].values
print(f'Chargement GTFS : {len(stops_df)} arrêts')
print()

for pop_size in POPULATION_SIZES:
    fname      = pop_filename(pop_size)
    fixed_path = TEMP_FIXED / fname
    pt_path    = TEMP_PT    / fname

    if not should_force('pt_enriched') and pt_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/3_pt_enriched/')
        continue

    if not fixed_path.exists():
        raise FileNotFoundError(f'Fichier fixed manquant — relancer étape 2 : {fixed_path}')

    data = load_json(fixed_path)
    n = enrich_public_transport(data, stop_lats, stop_lons, MAX_PT_DIST_M)
    save_json(data, pt_path)
    print(f'[OK]   {fname} — {n} localisation(s) enrichie(s) → Temp/3_pt_enriched/')
    n_p, n_e = check_temporal_order(data)
    print(f'       {"[OK]  " if n_e == 0 else "[WARN]"} Ordre temporel : {len(data) - n_p}/{len(data)} personnes valides')

print()
print('Étape 3 terminée.')

---
## Étape 3bis — Enrichissement zone urbaine/périurbaine/rurale → `Temp/4_zone_enriched/`

Pour chaque localisation d'activité, on détermine le type de zone à partir de deux tables INSEE :

| Source | Rôle |
|---|---|
| **AAV 2020** (Zonage en Aires d'Attraction des Villes) | Classification pôle / couronne / hors influence |
| **Grille de densité communale** | Niveau de densité 1–4 (dense → très peu dense) |

**Pipeline** :
1. Collecte de toutes les coordonnées uniques de la population
2. Géocodage inverse batch via l'API BAN (`api-adresse.data.gouv.fr`) → code INSEE commune
3. Jointure avec AAV2020 + grille de densité
4. Ajout du champ `zone_type` dans chaque `location` d'activité

**Valeurs possibles de `zone_type`** :

| Valeur | Signification |
|---|---|
| `urbain_dense` | Pôle AAV + commune dense ou intermédiaire |
| `urbain` | Pôle AAV + commune peu dense |
| `periurbain_dense` | Couronne AAV + commune dense ou intermédiaire |
| `periurbain` | Couronne/multipolarisé + commune peu dense |
| `bourg` | Hors influence AAV + commune dense |
| `rural` | Hors influence AAV + commune peu dense |
| `inconnu` | Code commune non trouvé dans les tables INSEE |

> **Prérequis** : placer dans `data/insee/` les fichiers `AAV2020_au_01-01-2023.csv` et `grille_communale_densite_2023.csv` (voir section "Données INSEE requises" en introduction).

In [ ]:
# ── Étape 3bis — Enrichissement zone (AAV2020 + densité) → Temp/4_zone_enriched/ ─
import csv
import io
import ssl
import time as _time

INSEE_DIR          = REPO_ROOT / 'data' / 'insee'
DENSITE_FILE       = INSEE_DIR / 'fichier_diffusion_2026.xlsx'
AAV_FILE           = INSEE_DIR / 'AAV2020_au_01-01-2026.xlsx'
GEOCODE_CACHE_FILE = TEMP_DIR  / 'geocode_cache.json'

_SSL_CTX = ssl.create_default_context()
_SSL_CTX.check_hostname = False
_SSL_CTX.verify_mode    = ssl.CERT_NONE

print('=' * 60)
print('ÉTAPE 3bis — Enrichissement zone → Temp/4_zone_enriched/')
print('=' * 60)

if should_force('zone_enriched') and GEOCODE_CACHE_FILE.exists():
    GEOCODE_CACHE_FILE.unlink()
    print('[RESET] Cache géocodage supprimé (FORCE_STEP)')

if not DENSITE_FILE.exists():
    raise FileNotFoundError(
        f'Fichier densité manquant : {DENSITE_FILE}\n'
        f'Télécharger depuis https://www.insee.fr/fr/statistiques/5040028\n'
        f'et placer dans {INSEE_DIR}/'
    )

# ── Chargement grille de densité (DENS7) ─────────────────────────────────────
densite = pd.read_excel(
    DENSITE_FILE, sheet_name='Maille communale', header=4,
    dtype={'CODGEO': str}, engine='calamine',
)
densite.columns = [c.strip() for c in densite.columns]
print(f'Grille densité : {len(densite)} communes')

# Libellés naturels pour DENS7 — utilisés dans la phrase finale
DENS7_LABEL = {
    1: 'grand centre urbain',
    2: 'centre urbain intermédiaire',
    3: 'ceinture urbaine',
    4: 'petite ville',
    5: 'bourg rural',
    6: 'habitat rural dispersé',
    7: 'habitat rural très dispersé',
}
densite['_detail'] = densite['DENS7'].map(DENS7_LABEL).fillna('zone inconnue')
_densite_lookup    = densite.set_index('CODGEO')['_detail'].to_dict()

print('Distribution DENS7 :')
print(densite['_detail'].value_counts().to_string())
print()

# ── Chargement AAV2020 — ville-centre ─────────────────────────────────────────
_attraction_lookup: dict[str, str] = {}
if AAV_FILE.exists():
    try:
        aav = pd.read_excel(
            AAV_FILE, sheet_name='Composition_communale',
            header=5, dtype={'CODGEO': str}, engine='calamine',
        )
        aav.columns = [c.strip() for c in aav.columns]
        aav['_city'] = aav['LIBAAV2020'].where(aav['CATEAAV2020'] != '30', other='')
        _attraction_lookup = aav.set_index('CODGEO')['_city'].to_dict()
        print(f'AAV2020 : {len(_attraction_lookup)} communes')
    except Exception as e:
        print(f'[WARN] AAV2020 non chargé ({e})')
else:
    print('[INFO] AAV2020 absent — ville-centre ignorée')
print()

def _build_zone_label(codgeo: str) -> str:
    """
    Exemples :
      31555 → 'quartier de grand centre urbain sur la commune de Toulouse'
      31561 → 'quartier de ceinture urbaine sur la commune de Tournefeuille (aire de Toulouse)'
      09001 → 'bourg rural hors aire d'attraction urbaine'
    """
    label = _densite_lookup.get(codgeo, 'zone inconnue')
    city  = _attraction_lookup.get(codgeo, '')
    if not city:
        return f'{label} hors aire d\'attraction urbaine'
    return f'quartier de {label} sur la commune de {city}'

# ── Géocodage inverse batch (API BAN) — multipart/form-data ──────────────────
def _make_multipart(csv_bytes: bytes) -> tuple[bytes, str]:
    boundary = f'----BoundaryZone{int(_time.time())}'
    body = (
        f'--{boundary}\r\n'
        f'Content-Disposition: form-data; name="data"; filename="coords.csv"\r\n'
        f'Content-Type: text/csv\r\n\r\n'
    ).encode() + csv_bytes + f'\r\n--{boundary}--\r\n'.encode()
    return body, f'multipart/form-data; boundary={boundary}'

_geocode_cache: dict[tuple, str] = {}
if GEOCODE_CACHE_FILE.exists():
    _raw = load_json(GEOCODE_CACHE_FILE)
    _geocode_cache = {(float(k.split(',')[0]), float(k.split(',')[1])): v
                      for k, v in _raw.items()}
    print(f'Cache géocodage : {len(_geocode_cache)} coords')

def _save_geocode_cache() -> None:
    save_json({f'{k[0]},{k[1]}': v for k, v in _geocode_cache.items()}, GEOCODE_CACHE_FILE)

def _batch_reverse_geocode(coords: list[tuple[float, float]]) -> dict[tuple, str]:
    BATCH   = 5000
    result  = dict(_geocode_cache)
    missing = [c for c in coords if c not in result]
    if not missing:
        return result
    print(f'  {len(missing)} coords à géocoder ({len(coords) - len(missing)} en cache)…')
    for start in range(0, len(missing), BATCH):
        chunk     = missing[start:start + BATCH]
        csv_bytes = ('longitude,latitude\n' + '\n'.join(f'{lon},{lat}' for lon, lat in chunk)).encode()
        body, ctype = _make_multipart(csv_bytes)
        req = urllib.request.Request(
            'https://api-adresse.data.gouv.fr/reverse/csv/',
            data=body, headers={'Content-Type': ctype}, method='POST',
        )
        try:
            with urllib.request.urlopen(req, timeout=120, context=_SSL_CTX) as resp:
                reader = csv.DictReader(io.TextIOWrapper(resp, encoding='utf-8'))
                for row in reader:
                    try:
                        key = (float(row['longitude']), float(row['latitude']))
                        result[key] = row.get('result_citycode', '')
                        _geocode_cache[key] = result[key]
                    except (KeyError, ValueError):
                        pass
        except Exception as e:
            print(f'  [WARN] BAN API erreur (chunk {start}) : {e}')
        _save_geocode_cache()
        print(f'  Géocodé {min(start + BATCH, len(missing))}/{len(missing)} coords…')
    n_ok = sum(1 for v in result.values() if v)
    print(f'  {n_ok}/{len(coords)} codes commune résolus')
    return result

_STALE_FIELDS = {'zone_city', 'zone_density', 'zone_type', 'zone_type_detail',
                 'commune_pop', 'attraction_city', 'aav_category'}

# ── Traitement par taille de population ───────────────────────────────────────
for pop_size in POPULATION_SIZES:
    fname     = pop_filename(pop_size)
    pt_path   = TEMP_PT   / fname
    zone_path = TEMP_ZONE / fname

    if not should_force('zone_enriched') and zone_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/4_zone_enriched/')
        continue

    if not pt_path.exists():
        raise FileNotFoundError(f'Fichier pt_enriched manquant — relancer étape 3 : {pt_path}')

    data = load_json(pt_path)

    unique_locs = list({
        (act['location']['lon'], act['location']['lat'])
        for person in data
        for act in person.get('identity', {}).get('activities', [])
        if act.get('location') and act['location'].get('lon') is not None
    })
    print(f'{fname} : {len(unique_locs)} locations uniques à géocoder…')

    loc_to_codgeo = _batch_reverse_geocode(unique_locs)
    loc_to_zone   = {loc: _build_zone_label(codgeo) for loc, codgeo in loc_to_codgeo.items()}

    n_enriched = 0
    for person in data:
        for act in person.get('identity', {}).get('activities', []):
            loc = act.get('location')
            if not loc or loc.get('lon') is None:
                continue
            for f in _STALE_FIELDS:
                loc.pop(f, None)
            loc['zone'] = loc_to_zone.get((loc['lon'], loc['lat']), 'zone inconnue')
            n_enriched += 1

    save_json(data, zone_path)

    zone_counts: dict[str, int] = {}
    for person in data:
        for act in person.get('identity', {}).get('activities', []):
            z = act.get('location', {}).get('zone', 'zone inconnue')
            zone_counts[z] = zone_counts.get(z, 0) + 1
    print(f'[OK]   {fname} — {n_enriched} activités enrichies → Temp/4_zone_enriched/')
    top5 = sorted(zone_counts.items(), key=lambda x: -x[1])[:5]
    print(f'       Top 5 zones : {top5}')

print()
print('Étape 3bis terminée.')

---
## Étape 4 — Calcul des temps de trajet (scheduling) + Ajustement des horaires → `Temp/5_scheduled/`

Pour chaque personne, on calcule **un seul temps de trajet** par paire d'activités :
- `car` pour les propriétaires de voiture
- `bicycle` pour les autres

Ce temps est passé directement à `ajuster_planning` (paramètre `travel_times`) — il n'est **pas stocké dans le JSON**.
Le cache SQLite OSMnx (foot / bicycle / car × 24h) est alimenté séparément lors du warm-up serveur.

- **Déduplication globale** : les paires communes à plusieurs tailles ne sont calculées qu'une seule fois.
- **Résultat** : plannings cohérents écrits directement dans `Temp/5_scheduled/`.


In [ ]:
# ── Étape 4+5 — Temps de trajet scheduling + Ajustement des horaires ─────────
import multiprocessing

from population_utils import (
    collect_scheduling_pairs, build_travel_times, ajuster_planning,
)
from route_worker import init_worker, compute_route_worker

print('=' * 60)
print('ÉTAPES 4+5 — Scheduling routes + Ajustement des horaires')
print('=' * 60)

to_schedule = []
for pop_size in POPULATION_SIZES:
    fname          = pop_filename(pop_size)
    scheduled_path = TEMP_SCHEDULED / fname

    if not should_force('scheduled') and scheduled_path.exists():
        print(f'[SKIP] {fname} — déjà dans Temp/5_scheduled/')
        continue

    zone_path = TEMP_ZONE / fname
    if not zone_path.exists():
        raise FileNotFoundError(f'Fichier zone_enriched manquant — relancer étape 3bis : {zone_path}')

    to_schedule.append(pop_size)

if not to_schedule:
    print()
    print('Tous les fichiers sont déjà planifiés — étape ignorée.')
else:
    all_data: dict[int, list] = {}
    for pop_size in to_schedule:
        all_data[pop_size] = load_json(TEMP_ZONE / pop_filename(pop_size))

    # ── Collecte des paires de scheduling (1 mode par personne) ──────────────
    print()
    print('Collecte des paires de scheduling…')
    global_pairs: set = set()
    per_size_pairs: dict[int, set] = {}
    for pop_size, data in all_data.items():
        pairs = collect_scheduling_pairs(data)
        per_size_pairs[pop_size] = pairs
        global_pairs |= pairs
        print(f'  toulouse_population_{pop_size} : {len(pairs)} paires (scheduling mode)')

    print(f'Total unique global : {len(global_pairs)} paires à calculer')
    print()

    route_cache: dict[tuple, dict | None] = {}

    if global_pairs:
        # bicycle/foot : hour=None → passer 8h (résultat identique quelle que soit l'heure)
        tasks = [
            (lat1, lon1, lat2, lon2, mode, hour if hour is not None else 8)
            for lat1, lon1, lat2, lon2, mode, hour in global_pairs
        ]

        print(f'Calcul de {len(tasks)} routes avec {MAX_WORKERS} workers…')
        t0 = time.monotonic()

        ctx = multiprocessing.get_context('spawn')
        with ProcessPoolExecutor(
            max_workers=MAX_WORKERS,
            mp_context=ctx,
            initializer=init_worker,
            initargs=(LLMAGENTS_PATH, str(OSMNX_CACHE), CACHE_KEY),
        ) as pool:
            results = list(tqdm(
                pool.map(compute_route_worker, tasks, chunksize=20),
                total=len(tasks),
                desc='scheduling routes',
            ))

        elapsed = time.monotonic() - t0
        print(f'Terminé en {elapsed:.1f}s  ({elapsed / len(tasks) * 1000:.1f} ms/route)')

        ok_count = null_count = 0
        for args, result in results:
            lat1, lon1, lat2, lon2, mode, actual_hour = args
            route_cache[(lat1, lon1, lat2, lon2, mode, actual_hour if mode == 'car' else None)] = result
            if result is None:
                null_count += 1
            else:
                ok_count += 1
        print(f'Routes : {ok_count} OK, {null_count} inaccessibles (None)')
        print()

    # ── Build travel_times + ajuster_planning + sauvegarde ───────────────────
    for pop_size in to_schedule:
        fname = pop_filename(pop_size)
        data  = all_data[pop_size]

        travel_times_list = build_travel_times(data, route_cache)

        sched_errors = 0
        for entry, tt in zip(data, travel_times_list):
            acts = entry.get('identity', {}).get('activities', [])
            try:
                entry['identity']['activities'] = ajuster_planning(
                    fname, entry.get('person_id', '?'), acts,
                    travel_times=tt, raise_error=True,
                )
            except ValueError as exc:
                sched_errors += 1

        save_json(data, TEMP_SCHEDULED / fname)
        n_p, n_e = check_temporal_order(data)
        status = '[OK]  ' if n_e == 0 else '[WARN]'
        print(f'[OK]   {fname} — {len(data)} personnes → Temp/5_scheduled/')
        print(f'       {status} Ordre temporel : {len(data) - n_p}/{len(data)} valides')
        if sched_errors:
            print(f'       [WARN] {sched_errors} conflits de planning non résolus')

print()
print('Étapes 4+5 terminées.')

---
## Étape 5 — *(fusionnée avec l'étape 4)*

L'ajustement des horaires est désormais effectué directement dans l'étape 4.
Cette cellule est conservée pour la compatibilité du pipeline mais n'exécute rien.


In [ ]:
# Étape 5 fusionnée avec étape 4 — rien à faire ici.
print("Étape 5 — déjà effectuée dans l'étape 4.")


---
## Étape 6 — Warm-up SQLite OSMnx → `data/osmnx_cache/toulouse_population_N/`

Pour chaque paire O-D unique de la population, on calcule **toutes les routes** nécessaires au serveur :
- `foot` × 1 entrée (indépendant du temps)
- `bicycle` × 1 entrée (indépendant du temps)
- `car` × 24 entrées horaires (congestion TomTom par plage)

Les résultats sont écrits dans le cache SQLite persistant (`osmnx_cache.db`).
Au démarrage du serveur, `get_direct_plan()` trouvera **100 % de hits** sur ce cache.

La cellule est **idempotente** : les routes déjà présentes en base sont ignorées.


In [ ]:
# ── Étape 6 — Warm-up SQLite OSMnx ──────────────────────────────────────────
import multiprocessing
from datetime import datetime as _dt, date as _date

from population_utils import collect_warmup_pairs
from route_worker import init_worker, compute_route_worker
from trip_helper.osmnx_persistent_cache import OsmnxPersistentCache

_SIM_DATE = _date(2024, 1, 8)  # lundi — doit correspondre au jour de simulation

print('=' * 60)
print('ÉTAPE 6 — Warm-up SQLite OSMnx (foot + bicycle + car × 24h)')
print('=' * 60)

for pop_size in POPULATION_SIZES:
    fname          = pop_filename(pop_size)
    scheduled_path = TEMP_SCHEDULED / fname

    if not scheduled_path.exists():
        print(f'[SKIP] {fname} — fichier scheduled manquant, relancer étape 4+5')
        continue

    data      = load_json(scheduled_path)
    all_pairs = collect_warmup_pairs(data)  # foot/bicycle × 1 + car × 24h

    _cache_dir = OSMNX_ROUTE_CACHE / f'toulouse_population_{pop_size}'
    _sqlite    = OsmnxPersistentCache(str(_cache_dir))

    # ── Vérification de couverture (idempotence) ───────────────────────────────
    pair_list = list(all_pairs)
    missing   = []
    for lat1, lon1, lat2, lon2, mode, hour in pair_list:
        actual_h  = hour if hour is not None else 8
        cdt       = _dt(_SIM_DATE.year, _SIM_DATE.month, _SIM_DATE.day, actual_h, 0)
        key, *_   = OsmnxPersistentCache.make_key(cdt, mode, lat1, lon1, lat2, lon2)
        if not _sqlite.lookup(key).found:
            missing.append((lat1, lon1, lat2, lon2, mode, hour))

    print(f'\n{fname}: {len(all_pairs)} paires total — {len(missing)} à calculer')

    if not missing:
        print(f'  [OK] Cache SQLite déjà complet.')
        continue

    # ── Calcul des routes manquantes ──────────────────────────────────────────
    tasks = [
        (lat1, lon1, lat2, lon2, mode, hour if hour is not None else 8)
        for lat1, lon1, lat2, lon2, mode, hour in missing
    ]

    print(f'  Calcul de {len(tasks)} routes avec {MAX_WORKERS} workers…')
    t0 = time.monotonic()

    ctx = multiprocessing.get_context('spawn')
    with ProcessPoolExecutor(
        max_workers=MAX_WORKERS,
        mp_context=ctx,
        initializer=init_worker,
        initargs=(LLMAGENTS_PATH, str(OSMNX_CACHE), CACHE_KEY),
    ) as pool:
        results = list(tqdm(
            pool.map(compute_route_worker, tasks, chunksize=20),
            total=len(tasks),
            desc=f'warmup {pop_size}',
        ))

    elapsed = time.monotonic() - t0
    print(f'  Terminé en {elapsed:.1f}s  ({elapsed / len(tasks) * 1000:.1f} ms/route)')

    # ── Écriture en SQLite ─────────────────────────────────────────────────────
    n_ok = n_null = 0
    for (lat1, lon1, lat2, lon2, mode, hour), (_, result) in zip(missing, results):
        actual_h              = hour if hour is not None else 8
        cdt                   = _dt(_SIM_DATE.year, _SIM_DATE.month, _SIM_DATE.day, actual_h, 0)
        key, date_s, dow, bkt = OsmnxPersistentCache.make_key(cdt, mode, lat1, lon1, lat2, lon2)
        _sqlite.store(key, date_s, dow, bkt, mode, lat1, lon1, lat2, lon2, result)
        if result is None:
            n_null += 1
        else:
            n_ok += 1

    print(f'  SQLite : {n_ok} OK, {n_null} inaccessibles → {_cache_dir.relative_to(REPO_ROOT)}/osmnx_cache.db')

print()
print('Étape 6 terminée.')


---
## Export final → `data/eqasim_output/`

Copie des fichiers planifiés depuis `Temp/5_scheduled/` vers le dossier de sortie définitif `data/eqasim_output/`. Un bilan de qualité est affiché pour chaque taille avant la copie :

| Indicateur | Signification |
|---|---|
| `missing_routes` | Agents dont au moins un trajet n'a pas pu être calculé (zones inaccessibles) |
| `missing_any_mode` | Agents dont le mode de transport reste indéterminé |
| `missing_pt` | Agents rattachés aux TC mais sans arrêt Tisséo dans le rayon de 1 500 m |

Les fichiers exportés sont directement consommables par la simulation GAMA et le serveur d'agents LLM.

In [ ]:
# check_enrichment supprimé — les routes ne sont plus dans le JSON
for pop_size in POPULATION_SIZES:
    fname = pop_filename(pop_size)
    path = TEMP_SCHEDULED / fname
    if not path.exists():
        print(f'[MISSING] {fname}')
        continue
    data = load_json(path)
    n_p, n_e = check_temporal_order(data)
    print(f'{fname}: {len(data)} personnes, {len(data)-n_p}/{len(data)} plannings valides')
